# DistilBERT — Fake News Detection (LIAR + Kaggle)

Step 4 of the project pipeline. Trains **DistilBERT only** first (faster, fits free-tier GPU, good for debugging before BERT/RoBERTa).

**Before running:** Runtime → Change runtime type → GPU (T4 is fine).

Upload `liar_clean.csv` and `kaggle_clean.csv` to this Colab session (left sidebar → Files → upload), or mount Google Drive.

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas

## Imports and setup

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

MODEL_NAME = "distilbert-base-uncased"
LABEL_MAP = {"FAKE": 0, "REAL": 1}
MAX_LENGTH = {"liar": 64, "kaggle": 256}  # LIAR = short claims, Kaggle = full articles

## Reusable training function

In [ ]:
def load_data(csv_path):
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["transformer_text", "binary_label"])
    df["label"] = df["binary_label"].map(LABEL_MAP)
    return df[["transformer_text", "label"]].rename(columns={"transformer_text": "text"})


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, pos_label=0),  # FAKE = 0
        "recall": recall_score(labels, preds, pos_label=0),
        "f1": f1_score(labels, preds, pos_label=0),
    }


def train_distilbert(dataset_name, csv_path, epochs=3, batch_size=16, max_samples=None):
    print(f"\n{'='*60}\nTraining DistilBERT on {dataset_name}\n{'='*60}")

    max_length = MAX_LENGTH[dataset_name]
    df = load_data(csv_path)
    if max_samples is not None and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42).reset_index(drop=True)
        print(f"Subsampled to {max_samples} rows to keep runtime manageable")
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
    print(f"Train: {len(train_df)} | Test: {len(test_df)}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    def tokenize(batch):
        return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=max_length)

    train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize, batched=True)
    test_ds = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize, batched=True)

    training_args = TrainingArguments(
        output_dir=f"./results_{dataset_name}_distilbert",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=50,
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=test_ds,
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    training_time_sec = time.time() - start

    metrics = trainer.evaluate()

    preds = trainer.predict(test_ds)
    pred_labels = np.argmax(preds.predictions, axis=-1)
    cm = confusion_matrix(test_ds["label"], pred_labels, labels=[0, 1])

    print(f"\nTraining time: {training_time_sec/60:.1f} minutes")
    print(f"Accuracy:  {metrics['eval_accuracy']:.4f}")
    print(f"Precision: {metrics['eval_precision']:.4f}")
    print(f"Recall:    {metrics['eval_recall']:.4f}")
    print(f"F1 score:  {metrics['eval_f1']:.4f}")
    print(f"Confusion matrix [rows=true, cols=pred, order=FAKE/REAL]:\n{cm}")

    model.save_pretrained(f"./saved_model_{dataset_name}_distilbert")
    tokenizer.save_pretrained(f"./saved_model_{dataset_name}_distilbert")

    return {
        "dataset": dataset_name, "model": "DistilBERT",
        "accuracy": metrics["eval_accuracy"], "precision": metrics["eval_precision"],
        "recall": metrics["eval_recall"], "f1": metrics["eval_f1"],
        "training_time_min": round(training_time_sec / 60, 1),
    }

## Run: DistilBERT on LIAR

In [ ]:
liar_result = train_distilbert("liar", "liar_clean.csv")

## Run: DistilBERT on Kaggle

Expect this to take longer than LIAR — full articles, longer sequences (256 tokens vs 64).

In [ ]:
kaggle_result = train_distilbert("kaggle", "kaggle_clean.csv", max_samples=15000)

## Compare against your baseline table

In [ ]:
# Paste your baseline results here (from results_baselines.csv, Step 3)
baseline_results = [
    {"dataset": "liar",   "model": "Logistic Regression", "accuracy": 0.6036, "precision": 0.5700, "recall": 0.4184, "f1": 0.4826},
    {"dataset": "liar",   "model": "Naive Bayes",          "accuracy": 0.6048, "precision": 0.6035, "recall": 0.3076, "f1": 0.4075},
    {"dataset": "liar",   "model": "Linear SVM",           "accuracy": 0.5781, "precision": 0.5254, "recall": 0.4672, "f1": 0.4946},
    {"dataset": "kaggle", "model": "Logistic Regression", "accuracy": 0.9830, "precision": 0.9870, "recall": 0.9757, "f1": 0.9813},
    {"dataset": "kaggle", "model": "Naive Bayes",          "accuracy": 0.9375, "precision": 0.9316, "recall": 0.9319, "f1": 0.9318},
    {"dataset": "kaggle", "model": "Linear SVM",           "accuracy": 0.9891, "precision": 0.9916, "recall": 0.9846, "f1": 0.9881},
]

all_results = pd.DataFrame(baseline_results + [liar_result, kaggle_result])
all_results.to_csv("results_full_comparison.csv", index=False)
all_results.sort_values(["dataset", "f1"], ascending=[True, False])

## Download results

Run the cell below, then bring `results_full_comparison.csv` (and the `saved_model_*` folders, if you want the weights) back to continue with Step 5 (SHAP/LIME explainability).

In [ ]:
from google.colab import files
files.download("results_full_comparison.csv")